In [1]:
# Setup dataset: Chua circuit (recall). Same structure as Thomas: chaos + two coexisting mirror-image orbits.
import numpy as np
from dynamics_testing import sim, attractor_dist, make_train

SEG, SETTLE, REENTRAIN, N_AUTO, DISCARD, WARM_TRAIN = 2000, 1000, 50, 1200, 200, 200
STATES = ["chaotic", "periodic_A", "periodic_B"]
RS_ROT = 700
N_STEPS = SETTLE + RS_ROT + N_AUTO
DT = 0.05                                   # ~29 frames per cycle (Thomas chaos: ~30)
A_CHAOS, A_PER = 15.6, 12.0                 # double scroll | pair of asymmetric limit cycles
M0, M1, BETA = -1.143, -0.714, 28.0

def rhs(_, y, al):                          # time rescaled by DT: dynamics_testing's unit sampling = one frame
    x, yy, z = y
    g = M1 * x + 0.5 * (M0 - M1) * (abs(x + 1) - abs(x - 1))
    return [DT * al * (yy - x - g), DT * (x - yy + z), -DT * BETA * yy]

SCALE = 4.0 / np.abs(sim(rhs, A_CHAOS, 20_000, (0.1, 0.0, 0.0))).max()   # odd-symmetric: scale only, no centering
norm = lambda y: y * SCALE
ic = lambda rng: rng.uniform(-0.5, 0.5, 3)                                  # large ICs can escape (outer slope M1 < 0)

def sim_n(al, n, init):
    y = norm(sim(rhs, al, n, init))
    return y if np.all(np.isfinite(y)) and np.abs(y).max() < 10 else None

TRAIN = norm(make_train(rhs, SEG, b1=A_CHAOS, b2=A_PER, init=(0.1, 0.0, 0.0)))

def make_states(n_ics, n_steps, seed):
    rng = np.random.default_rng(seed)
    out = {s: [] for s in STATES}
    while min(len(v) for v in out.values()) < n_ics:
        tr = sim_n(A_CHAOS, n_steps, ic(rng))
        if tr is not None and len(out["chaotic"]) < n_ics:
            out["chaotic"].append(tr)
        tr = sim_n(A_PER, n_steps, ic(rng))
        if tr is not None:
            k = "periodic_A" if tr[0].mean() > 0 else "periodic_B"     # orbits are mirror images in x
            if len(out[k]) < n_ics:
                out[k].append(tr)
    return out

REF = {k: np.concatenate(v, axis=1) for k, v in make_states(2, 5000, seed=300).items()}
VAL, TEST = make_states(4, N_STEPS, seed=100), make_states(4, N_STEPS, seed=200)

print("train segment mean x:", [round(TRAIN[0, k * SEG:(k + 1) * SEG].mean(), 2) for k in range(4)])  # both orbits present
print({f"{a}-{b}": round(attractor_dist(REF[a], REF[b]), 3) for a in STATES for b in STATES if a < b})

train segment mean x: [np.float64(0.0), np.float64(-1.44), np.float64(0.12), np.float64(1.42)]
{'chaotic-periodic_A': np.float64(1.008), 'chaotic-periodic_B': np.float64(0.944), 'periodic_A-periodic_B': np.float64(1.901)}


In [2]:
# Setup model testing harnesses
from testing_harness import get_harness

RS_OMEGA0 = 0.01
RN_BASE = dict(input_dims=3, spectral_radius=0.6, input_weight=1.2, average_degree_nodes=10, num_nodes=100,
               leakage=0, omega0=RS_OMEGA0, epsilon1=-0.2, epsilon2=0.6,
               mean_phase_tolerance=RS_OMEGA0,          # >= one rotation step, so the pi-crossing can't be skipped
               error_tolerance=np.inf)                  # freeze on phase alone
SEEDS = list(range(8))
HARNESS = get_harness()

In [3]:
# Spiking, RS, ESN recall comparison
import os, time
import pandas as pd
from dynamics_testing import valid_time, classify
from testing_harness import ALPHAS, run_frozen
from scipy.stats import binomtest

MODELS = {"spike_c1.0": ("spk", 1.0),
          "spike_c0":   ("spk", 0.0),     # ESN control sharing W, W_in with spike_c1.0
          "RN_pub":     ("rs",  0.6),
          "RN_c0":      ("rs",  0.0)}     # ESN control sharing W, W_in with RN_pub
OUT = "recall_vs_rs-chua.csv"
done = set(map(tuple, pd.read_csv(OUT)[["model", "seed"]].drop_duplicates().values)) if os.path.exists(OUT) else set()
# done = {d for d in done if not d[0].startswith("RN")}
print(done)

for name, (kind, c) in MODELS.items():
    train_fn, settle_fn = HARNESS[kind]
    for seed in SEEDS:
        if (name, seed) in done:
            continue
        t0, rows = time.time(), []
        net, W = train_fn(RN_BASE, seed, c, TRAIN, WARM_TRAIN)
        for split, sets in (("val", VAL), ("test", TEST)):
            for state, trajs in sets.items():
                for i, x in enumerate(trajs):
                    warm, t, n_re, froze, cost = settle_fn(net, x, SETTLE, REENTRAIN, offset=1 + i)
                    for a in ALPHAS:                                # all alphas logged; chosen on val afterwards
                        if name.startswith("RN"):
                            net.output_weights = W[a]
                            p, ts = net.predict_search(x, SETTLE, REENTRAIN)
                        else:
                            p, ts = run_frozen(warm, x, t, W[a], n_re, N_AUTO)
                        label, _ = classify(p[:, DISCARD:], REF)
                        rows.append(dict(model=name, seed=seed, split=split, alpha=a, state=state, ic=i,
                                         pred=label, correct=label == state, vpt=valid_time(p, x[:, ts:]),
                                         froze=froze, **{k: cost[k] for k in
                                         ("event_msgs", "dense_msgs", "msg_ratio", "rate_mean", "frac_silent")}))
        pd.DataFrame(rows).to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)
        print(f"{name} seed {seed}: {time.time() - t0:.0f}s", flush=True)

# ======================= analysis =======================
df = pd.read_csv(OUT); df["correct"] = df.correct.astype(bool)

df = df.drop_duplicates(["model", "seed", "split", "alpha", "state", "ic"], keep="last")

def scores(g):
    acc = g.groupby("state").correct.mean()
    per = g[g.state != "chaotic"]
    return pd.Series(dict(
        S=0.5 * acc["chaotic"] + 0.25 * acc["periodic_A"] + 0.25 * acc["periodic_B"],
        regime=0.5 * (g[g.state == "chaotic"].pred == "chaotic").mean()
               + 0.5 * per.pred.isin(["periodic_A", "periodic_B"]).mean(),
        orbit=(per.pred == per.state).mean()))

val = df[df.split == "val"]
BEST = (val.groupby(["model", "alpha"]).apply(scores).S
        .groupby("model").idxmax().map(lambda k: k[1]).to_dict())

test = df[(df.split == "test") & (df.alpha == df.model.map(BEST))]
print("alpha (val):", BEST)

acc = test.groupby(["model", "state"]).correct.agg(["sum", "count"])
print("Attractor likelihood (weighted)")
print((acc["sum"].astype(int).astype(str) + "/" + acc["count"].astype(str)).unstack())
print("Scores (unweighted)")
print(test.groupby("model").correct.mean().round(3).to_dict())
print("Scores (weighted)")
print(test.groupby("model").apply(scores, include_groups=False))
print("Per-seed scores (unweighted)")
print(test.groupby(["model", "seed"]).correct.mean().unstack().round(2))
print("RS freeze failures:", int((~test[test.model.str.startswith("RN")].froze.astype(bool)).sum()))

def paired(a, b, n_boot=10_000, rng=np.random.default_rng(0)):
    key = ["seed", "state", "ic"]
    A = test[test.model == a].set_index(key).correct
    B = test[test.model == b].set_index(key).correct
    A, B = A.align(B, join="inner")
    a_only, b_only = int((A & ~B).sum()), int((~A & B).sum())
    p = binomtest(a_only, a_only + b_only, 0.5).pvalue if a_only + b_only else 1.0
    d = (A.astype(int) - B.astype(int)).groupby(level="seed").mean()       # cluster bootstrap over seeds
    boot = [d.sample(len(d), replace=True, random_state=rng.integers(1 << 31)).mean() for _ in range(n_boot)]
    return dict(A=a, B=b, n=len(A), acc_A=A.mean(), acc_B=B.mean(), A_only=a_only, B_only=b_only,
                mcnemar_p=p, diff=d.mean(), ci_lo=np.percentile(boot, 2.5), ci_hi=np.percentile(boot, 97.5))

print(pd.DataFrame([paired("spike_c1.0", "RN_pub"),
                    paired("spike_c1.0", "spike_c0"),
                    paired("RN_pub", "RN_c0")]).round(3).to_string(index=False))

cost = test.drop_duplicates(["model", "seed", "state", "ic"])
print(cost.groupby("model")[["rate_mean", "frac_silent", "event_msgs", "dense_msgs", "msg_ratio"]].mean().round(4))

{('spike_c1.0', 1), ('spike_c1.0', 7), ('RN_c0', 4), ('RN_c0', 7), ('RN_pub', 3), ('RN_pub', 0), ('RN_pub', 6), ('spike_c0', 4), ('spike_c0', 1), ('spike_c0', 7), ('RN_c0', 0), ('spike_c1.0', 0), ('RN_c0', 3), ('spike_c1.0', 3), ('RN_c0', 6), ('spike_c1.0', 6), ('RN_pub', 2), ('RN_pub', 5), ('spike_c0', 0), ('spike_c0', 3), ('spike_c0', 6), ('RN_c0', 2), ('spike_c1.0', 2), ('RN_c0', 5), ('spike_c1.0', 5), ('RN_pub', 1), ('RN_pub', 4), ('RN_pub', 7), ('spike_c0', 2), ('spike_c0', 5), ('RN_c0', 1), ('spike_c1.0', 4)}
alpha (val): {'RN_c0': 0.001, 'RN_pub': 10.0, 'spike_c0': 0.001, 'spike_c1.0': 1e-05}
Attractor likelihood (weighted)
state      chaotic periodic_A periodic_B
model                                   
RN_c0         0/32      32/32      32/32
RN_pub        4/32      31/32      31/32
spike_c0      0/32      32/32      32/32
spike_c1.0   19/32      29/32      26/32
Scores (unweighted)
{'RN_c0': 0.667, 'RN_pub': 0.688, 'spike_c0': 0.667, 'spike_c1.0': 0.771}
Scores (weighted)
   

C:\Users\ianwh\AppData\Local\Temp\ipykernel_18004\2408037376.py:57: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  BEST = (val.groupby(["model", "alpha"]).apply(scores).S


         A        B  n  acc_A  acc_B  A_only  B_only  mcnemar_p  diff  ci_lo  ci_hi
spike_c1.0   RN_pub 96  0.771  0.688      19      11      0.200 0.083  0.010  0.167
spike_c1.0 spike_c0 96  0.771  0.667      19       9      0.087 0.104  0.052  0.156
    RN_pub    RN_c0 96  0.688  0.667       4       2      0.688 0.021 -0.021  0.073
            rate_mean  frac_silent  event_msgs  dense_msgs  msg_ratio
model                                                                
RN_c0             NaN          NaN         NaN     50000.0        NaN
RN_pub            NaN          NaN         NaN     50000.0        NaN
spike_c0       0.0747          0.0   3733.5375     50000.0     0.0747
spike_c1.0     0.0746          0.0   3731.8974     50000.0     0.0746


In [4]:
# Unsupervised detection: no labels, each detector calibrated on its own pre-switch window.
# Spiking and RS link layers need no W_out (training doesn't change link dynamics); the ESN detector does.
from tqdm import tqdm
import scipy.sparse as sparse
from testing_harness import RN
from spike_sharing import SpikingNetwork
from scipy.integrate import solve_ivp

N_PRE = N_POST = 2500                                # make_train uses equal segments
T_REF0, T_REF1, T_SW = 1000, 2000, N_PRE             # settle | reference | false-alarm window | switch
DET_ICS, TAU_S, PERSIST, KS = 4, 20, 10, (3, 5)
DIRS = {"chaos->periodic": (A_CHAOS, A_PER), "periodic->chaos": (A_PER, A_CHAOS)}

def make_streams(n_ics, seed):
    rng = np.random.default_rng(seed)
    streams = {d: [] for d in DIRS}
    for d, (a0, a1) in DIRS.items():
        while len(streams[d]) < n_ics:
            x = norm(make_train(rhs, N_PRE, b1=a0, b2=a1, n_seg=2, init=ic(rng)))   # same scaling as recall data
            if np.all(np.isfinite(x)) and np.abs(x).max() < 10:                    # reject escaped trajectories
                streams[d].append(x)
    return streams

STREAMS = make_streams(DET_ICS, seed=700)

def ema(s, tau):
    out, m = np.empty_like(s), s[0]
    for i, v in enumerate(s):
        m += (v - m) / tau; out[i] = m
    return out

def detect(s, k):
    s = ema(np.asarray(s, float), TAU_S)
    mu, sd = s[T_REF0:T_REF1].mean(), s[T_REF0:T_REF1].std() + 1e-12
    run = np.convolve((np.abs(s - mu) > k * sd).astype(int), np.ones(PERSIST, int))[:len(s)] >= PERSIST
    post = np.flatnonzero(run[T_SW:])
    return bool(run[T_REF1:T_SW].any()), (post[0] if len(post) else np.nan)

TAU_CTX = 60                                          # = spiking mod_tau, so all context signals share one filter

def ref_dist(X):                                      # RMS distance of each row from its reference-window mean
    return np.sqrt(np.mean(np.abs(X - X[T_REF0:T_REF1].mean(axis=0)) ** 2, axis=1))

def rel_phasor(phases):                               # e^{i(theta_i - psi)}, psi = global mean phase
    return np.exp(1j * (phases - np.angle(np.mean(np.exp(1j * phases)))))

det_rows, SIG = [], {}
for seed in SEEDS:
    t0 = time.time()
    spk = SpikingNetwork(model_seed=seed)                                  # final model, c = 1.0
    rs = RN(**RN_BASE, link_strength_change_ratio=0.6, model_seed=seed); rs.input_weights = spk.input_weights.copy()
    esn = RN(**RN_BASE, link_strength_change_ratio=0.0, model_seed=seed); esn.input_weights = spk.input_weights.copy()
    esn.train(TRAIN, warmup_time=WARM_TRAIN)                               # == spike_c0 (verified)
    esn.regularization, esn.lsqrs = BEST["spike_c0"], False; esn.compute_weights(); esn.reset_history()

    for d, xs in tqdm(STREAMS.items()):
        for i, x in enumerate(xs):
            T = x.shape[1]
            spk.reset_initial_states(1 + i); spk.reset_history()
            rs.reset_initial_states(seed_offset=1 + i); rs.reset_history()
            esn.reset_initial_states(seed_offset=1 + i); esn.reset_history()
            R_spk, R_rs, err = np.empty(T), np.empty(T), np.full(T, np.nan)
            ctx = np.empty((T, spk.num_links), np.complex64)
            ctx_rs = np.empty((T, rs.num_links), np.complex64)
            st_esn = np.empty((T, esn.num_nodes))
            z_rs, n_lp = None, None
            for t in range(T):
                spk.advance(x[:, t], save_history=False)
                rs.advance(x[:, t], save_history=False)
                esn.advance(x[:, t], save_history=False)
                r = rel_phasor(rs.link_states)
                z_rs = r if z_rs is None else z_rs + (r - z_rs) / TAU_CTX
                ctx_rs[t] = z_rs
                n_lp = esn.node_states.copy() if n_lp is None else n_lp + (esn.node_states - n_lp) / TAU_CTX
                st_esn[t] = n_lp
                R_spk[t] = np.abs(np.mean(np.exp(1j * spk.link_phase)))
                ctx[t] = spk.rel_lp
                R_rs[t] = np.abs(np.mean(np.exp(1j * rs.link_states)))
                if t + 1 < T:
                    err[t] = np.linalg.norm(esn.output_weights @ esn.node_states - x[:, t + 1])
            err[-1] = err[-2]
            ref = ctx[T_REF0:T_REF1].mean(axis=0)
            sig = {"spike_R": R_spk, "spike_ctx": ref_dist(ctx),
                   "RS_R": R_rs, "RS_ctx": ref_dist(ctx_rs),
                   "ESN_err": err, "ESN_state": ref_dist(st_esn)}
            SIG[(seed, d, i)] = sig
            for name, s in sig.items():
                for k in KS:
                    fa, lat = detect(s, k)
                    det_rows.append(dict(seed=seed, dir=d, ic=i, detector=name, k=k, false_alarm=fa, latency=lat))
    print(f"seed {seed}: {time.time() - t0:.0f}s", flush=True)

det = pd.DataFrame(det_rows); det.to_csv("detection-chua.csv", index=False); pd.to_pickle(SIG, "detection_signals-chua.pkl")
det["hit"] = ~det.false_alarm & det.latency.notna()
summ = det.groupby(["dir", "detector", "k"]).agg(hit=("hit", "mean"), false_alarm=("false_alarm", "mean"))
summ["lat_median"] = det[det.hit].groupby(["dir", "detector", "k"]).latency.median()
summ["lat_iqr"] = det[det.hit].groupby(["dir", "detector", "k"]).latency.agg(lambda v: v.quantile(.75) - v.quantile(.25))
print(summ.round(2).to_string())

100%|███████████████████████████████████████████████████████████████████████████████████| 2/2 [26:22<00:00, 791.26s/it]

seed 0: 1593s



100%|███████████████████████████████████████████████████████████████████████████████████| 2/2 [25:10<00:00, 755.22s/it]

seed 1: 1519s



100%|███████████████████████████████████████████████████████████████████████████████████| 2/2 [23:15<00:00, 697.97s/it]

...

seed 5: 1383s



100%|███████████████████████████████████████████████████████████████████████████████████| 2/2 [22:46<00:00, 683.36s/it]

seed 6: 1375s



100%|███████████████████████████████████████████████████████████████████████████████████| 2/2 [22:53<00:00, 686.94s/it]

seed 7: 1382s
                              hit  false_alarm  lat_median  lat_iqr
dir             detector  k                                        
chaos->periodic ESN_err   3  0.22         0.22        19.0     0.50
                          5  0.00         0.12         NaN      NaN
                ESN_state 3  0.50         0.12        63.0     8.25
                          5  0.00         0.00         NaN      NaN
                RS_R      3  0.56         0.25       132.5    49.00
                          5  0.19         0.00       280.5    82.00
                RS_ctx    3  0.38         0.62        82.5    36.25
                          5  1.00         0.00       167.0   130.75
                spike_R   3  0.00         0.25         NaN      NaN
                          5  0.00         0.00         NaN      NaN
                spike_ctx 3  0.44         0.16        79.0    23.25
                          5  0.00         0.00         NaN      NaN
periodic->chaos ESN_err   3  1.00 

In [5]:
SIG = pd.read_pickle("detection_signals-chua.pkl")
KGRID = np.round(np.arange(1, 12.01, 0.25), 2)
g = pd.DataFrame([dict(seed=s, dir=d, ic=i, detector=n, k=k, **dict(zip(("fa", "lat"), detect(x, k))))
                  for (s, d, i), sig in SIG.items() for n, x in sig.items() for k in KGRID])
g["hit"] = ~g.fa & g.lat.notna()
far = g.groupby(["detector", "k"]).fa.mean()                        # pooled over both directions: detector doesn't know which
K = {n: (f[f <= 0.05].index.get_level_values("k").min()) for n, f in far.groupby(level="detector")}
m = g[g.k == g.detector.map(K)]
res = m.groupby(["dir", "detector"]).agg(fa=("fa", "mean"), hit=("hit", "mean"))
res["lat_med"] = m[m.hit].groupby(["dir", "detector"]).lat.median()
print(K); print(res.round(2))

{'ESN_err': np.float64(7.75), 'ESN_state': np.float64(3.25), 'RS_R': np.float64(3.75), 'RS_ctx': np.float64(4.75), 'spike_R': np.float64(3.5), 'spike_ctx': np.float64(3.25)}
                             fa   hit  lat_med
dir             detector                      
chaos->periodic ESN_err    0.09  0.00      NaN
                ESN_state  0.00  0.50     69.5
                RS_R       0.03  0.53    184.0
                RS_ctx     0.03  0.97    183.0
                spike_R    0.00  0.00      NaN
                spike_ctx  0.09  0.47     89.0
periodic->chaos ESN_err    0.00  1.00      9.0
                ESN_state  0.00  1.00     31.5
                RS_R       0.06  0.94     52.0
                RS_ctx     0.06  0.94     74.5
                spike_R    0.00  0.75    872.0
                spike_ctx  0.00  1.00     48.0


In [6]:
from testing_harness import spk_settle, run_frozen
import copy

A_SW, N_STARTS, NICS = BEST["spike_c1.0"], 10, len(TEST["chaotic"])
reg = lambda s: "periodic" if str(s).startswith("periodic") else s
sw, bs = [], []

for seed in SEEDS:
    t0 = time.time()
    net = SpikingNetwork(model_seed=seed, link_strength_change_ratio=1.0, regularization=A_SW)
    net.train(TRAIN, warmup_time=WARM_TRAIN)
    W_out, H = net.output_weights.copy(), np.asarray(net.node_states_history).copy()   # H[t] <-> TRAIN[WARM_TRAIN + t]
    net.reset_history()
    seg = np.where(((WARM_TRAIN + np.arange(len(H))) // SEG) % 2 == 1, "train_periodic", "train_chaotic")

    warm, CTX = {}, {}
    for s in tqdm(STATES):
        for i, x in enumerate(TEST[s]):
            warm[(s, i)] = spk_settle(net, x, SETTLE, REENTRAIN, offset=1 + i)[0]
            CTX[(s, i)] = warm[(s, i)].rel_lp.copy()
    z0 = np.zeros_like(CTX[("chaotic", 0)])                       # neutral: m = 1 - c/2 on every link
    rng = np.random.default_rng(1000 + seed)

    # swap: drive and settle from one state, context implanted from another (different IC when same state)
    for d in tqdm(STATES):
        for i, x in enumerate(TEST[d]):
            conds = [("state", c, CTX[(c, (i + 1) % NICS if c == d else i)]) for c in STATES]
            conds += [("own", d, CTX[(d, i)]), ("shuffled", "none", rng.permutation(CTX[(d, i)])), ("neutral", "none", z0)]
            for kind, c, z in conds:
                w = copy.deepcopy(warm[(d, i)]); w.rel_lp = z.copy()
                p, _ = run_frozen(w, x, SETTLE, W_out, REENTRAIN, N_AUTO)
                lab = classify(p[:, DISCARD:], REF)[0]
                sw.append(dict(seed=seed, drive=d, ctx_kind=kind, ctx_state=c, pred=lab,
                               follows_ctx=lab == c, follows_drive=lab == d,
                               follows_ctx_regime=reg(lab) == reg(c)))

    # basin: closed loop from a node state under a fixed context, no drive at all
    for c, z in tqdm([(s, CTX[(s, i)]) for s in STATES for i in range(NICS)] + [("neutral", z0)]):
        for _ in range(N_STARTS):
            t = rng.integers(len(H))
            for start, n0 in ((seg[t], H[t]), ("gaussian", H.std() * rng.standard_normal(H.shape[1]))):
                w = copy.deepcopy(net); w.rel_lp, w.frozen, w.node_states = z.copy(), True, n0.copy()
                p, _ = run_frozen(w, None, 0, W_out, 0, N_AUTO)
                bs.append(dict(seed=seed, ctx=c, start=start, pred=classify(p[:, DISCARD:], REF)[0]))
    print(f"seed {seed}: {time.time() - t0:.0f}s", flush=True)

sw, bs = pd.DataFrame(sw), pd.DataFrame(bs)
sw.to_csv("swap-chua.csv", index=False); bs.to_csv("basin-chua.csv", index=False)

st = sw[sw.ctx_kind == "state"]; off = st[st.ctx_state != st.drive]
print(pd.crosstab([st.drive, st.ctx_state], st.pred))
print(off.groupby(["drive", "ctx_state"])[["follows_ctx", "follows_ctx_regime", "follows_drive"]].mean().round(2))
print(sw[sw.ctx_kind.isin(["own", "shuffled", "neutral"])].groupby(["ctx_kind", "drive"]).follows_drive.mean().unstack().round(2))

bs["ctx_group"] = bs.ctx.map(reg)
print(pd.crosstab([bs.start, bs.ctx], bs.pred, normalize="index").round(2))
ps = bs.assign(chaos=bs.pred == "chaotic").groupby(["seed", "ctx_group"]).chaos.mean().unstack()
sel = ps["chaotic"] - ps["periodic"]
boot = [sel.sample(len(sel), replace=True, random_state=k).mean() for k in range(10_000)]
print(ps.mean().round(2).to_dict(),
      f"| select = {sel.mean():.2f} [{np.percentile(boot, 2.5):.2f}, {np.percentile(boot, 97.5):.2f}]")

100%|██████████████████████████████████████████████████████████████████████████████████| 13/13 [01:00<00:00,  4.68s/it]

seed 0: 747s



100%|██████████████████████████████████████████████████████████████████████████████████| 13/13 [01:05<00:00,  5.05s/it]

seed 1: 750s



100%|██████████████████████████████████████████████████████████████████████████████████| 13/13 [01:04<00:00,  4.97s/it]

...


100%|██████████████████████████████████████████████████████████████████████████████████| 13/13 [01:01<00:00,  4.75s/it]

seed 6: 739s



100%|██████████████████████████████████████████████████████████████████████████████████| 13/13 [01:03<00:00,  4.86s/it]

seed 7: 731s


pred                   chaotic  none  periodic_A  periodic_B
drive      ctx_state                                        
chaotic    chaotic          25     0           2           5
           periodic_A        0     2           7          23
           periodic_B        3     3           8          18
periodic_A chaotic          23     1           7           1
           periodic_A        1     2          27           2
           periodic_B        2     1          27           2
periodic_B chaotic          23     1           0           8
           periodic_A        1     1           3          27
           periodic_B        3     1           2          26
                       follows_ctx  follows_ctx_regime  follows_drive
drive      ctx_state                                                 
chaotic    periodic_A         0.22                0.94           0.00
           periodic_B         0.56                0.81           0.09
periodic_A chaotic            0.72               